# SolarScan · Hito 2 · Comparación de modelos preentrenados de imágenes

En el Hito 1 comparamos EfficientNet-B0 contra un árbol de decisión sobre rasgos simples
(brillo, contraste, saturación). En el Hito 2 agregamos la comparación entre **modelos
preentrenados**, con exactamente la misma división de datos y el mismo protocolo de entrenamiento:

| Modelo | Familia | Parámetros aprox. |
|---|---|---|
| EfficientNet-B0 (modelo actual) | CNN | 5,3 M |
| ConvNeXt-Tiny | CNN moderna | 28 M |
| ViT-Small/16 | Transformer de visión (atención) | 22 M |

**Regla de decisión, fijada antes de entrenar:** se reemplaza el modelo en producción solo si otro
supera a EfficientNet-B0 por al menos **0,05 de macro-F1 en validación**. Motivo: con 118 fotos de
validación, diferencias menores a unos 5 puntos caen dentro del error de muestreo, y los candidatos
pesan 5 a 7 veces más y son más lentos en CPU (el servicio corre en Render sin GPU). La prueba se
usa una sola vez, para reportar. También se miden latencia en CPU y tamaño.

**Ejecución:** automática en GitHub Actions (`.github/workflows/modelos-hito2.yml`, CPU, cerca de
1,5 horas) o manual en Google Colab con GPU T4 (20 a 30 minutos). Los resultados quedan en
`notebooks/resultados_imagenes/`.

In [ ]:
!pip -q install timm==1.0.31 kagglehub==1.0.3

In [ ]:
import hashlib, json, random, time, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, torch, timm
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy.fft import dctn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix)

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEMILLA)
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
DISP = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DISP)
OUT = Path("resultados_imagenes"); OUT.mkdir(exist_ok=True)
CLASES = ['Bird-drop', 'Clean', 'Dusty', 'Electrical-damage', 'Physical-Damage', 'Snow-Covered']
CLASE_A_ID = {c: i for i, c in enumerate(CLASES)}
LABELS = list(range(len(CLASES)))

## 1. Datos y división (idéntica al Hito 1)
Se reutiliza la misma función de depuración: hashes, agrupación por similitud perceptual
(pHash, distancia ≤ 6) y reparto 70/15/15 por grupos con semilla 42. Al final se comprueba que
los conteos coinciden con los del Hito 1 (552 / 118 / 117).

In [ ]:
import kagglehub
DATA_ROOT = Path(kagglehub.dataset_download("pythonafroz/solar-panel-images"))
EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def preparar_dataset(data_root):
    """Misma lógica del notebook del Hito 1 (sin exportar láminas de contacto)."""
    data_root = Path(data_root); rows = []; bits = {}
    for p in sorted(data_root.rglob('*')):
        if not p.is_file() or p.suffix.lower() not in EXT: continue
        c = next((s for s in p.relative_to(data_root).parts[:-1] if s in CLASES), None)
        if c is None: continue
        i = len(rows); r = dict(id=i, ruta=str(p), clase=c, estado='valida')
        try:
            with Image.open(p) as im: im.verify()
            with Image.open(p) as im:
                im.load(); rgb = ImageOps.exif_transpose(im).convert('RGB')
                r['pixel_sha256'] = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
                d = dctn(np.asarray(rgb.convert('L').resize((32, 32)), dtype=float), type=2, norm='ortho')[:8, :8].flatten()
                bits[i] = d > np.median(d[1:])
        except Exception:
            r['estado'] = 'no_utilizable'
        rows.append(r)
    df = pd.DataFrame(rows); df['motivo'] = df.estado; df['split'] = 'excluida'; df['grupo'] = ''
    ids = df.index[df.estado.eq('valida')].tolist(); parent = {i: i for i in ids}
    def root(i):
        while parent[i] != i: parent[i] = parent[parent[i]]; i = parent[i]
        return i
    arr = np.stack([bits[i] for i in ids])
    for k, i in enumerate(ids):
        dist = np.count_nonzero(arr[k + 1:] != arr[k], axis=1)
        for off in np.flatnonzero(dist <= 6):
            j = ids[k + 1 + int(off)]; parent[root(j)] = root(i)
    for _, g in df.loc[ids].groupby('pixel_sha256'):
        for j in g.index[1:]: parent[root(j)] = root(g.index[0])
    for i in ids: df.at[i, 'grupo'] = f'g{root(i):04d}'
    for _, g in df.loc[ids].groupby('grupo'):
        if g.clase.nunique() > 1: df.loc[g.index, 'motivo'] = 'cuarentena_multiclase'; continue
        dup = g.duplicated('pixel_sha256')
        df.loc[g.index[dup], 'motivo'] = 'duplicado_exacto'; df.loc[g.index[~dup], 'motivo'] = 'incluida'
    rng = np.random.default_rng(42)
    for c in CLASES:
        sub = df[df.clase.eq(c) & df.motivo.eq('incluida')]
        groups = list(sub.groupby('grupo').groups.items()); rng.shuffle(groups); groups.sort(key=lambda g: -len(g[1]))
        objetivo = np.array([.7, .15, .15]) * len(sub); actual = np.zeros(3)
        for _, idx in groups:
            k = int(np.argmax(objetivo - actual)); df.loc[idx, 'split'] = ['train', 'val', 'test'][k]; actual[k] += len(idx)
    return df[df.motivo.eq('incluida')].copy()

datos = preparar_dataset(DATA_ROOT)
partes = {s: datos[datos.split.eq(s)].reset_index(drop=True) for s in ['train', 'val', 'test']}
conteo = {s: len(d) for s, d in partes.items()}
print("Conteo:", conteo)
assert conteo == {'train': 552, 'val': 118, 'test': 117}, "La división no coincide con el Hito 1"

## 2. Preprocesamiento y protocolo común
Igual que en el Hito 1: relleno gris a 224 × 224 conservando la proporción, volteo horizontal
y variación de brillo y contraste solo en entrenamiento, pérdida ponderada por clase, AdamW
(lr 1e-4), hasta 10 épocas con parada temprana de 3, y se conserva la época con mejor
macro-F1 de validación.

In [ ]:
class AjustarCuadrado:
    def __init__(self, tam=224): self.tam = tam
    def __call__(self, im):
        return ImageOps.pad(im, (self.tam, self.tam), method=Image.Resampling.BILINEAR, color=(128, 128, 128))

def abrir_rgb(ruta):
    with Image.open(ruta) as im:
        im.load(); return ImageOps.exif_transpose(im).convert('RGB')

class Paneles(Dataset):
    def __init__(self, df, tf): self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]; return self.tf(abrir_rgb(r.ruta)), CLASE_A_ID[r.clase]

def metricas(y, p):
    return {'accuracy': float(accuracy_score(y, p)), 'accuracy_balanceada': float(balanced_accuracy_score(y, p)),
            'macro_f1': float(f1_score(y, p, labels=LABELS, average='macro', zero_division=0))}

@torch.inference_mode()
def inferir(red, loader):
    red.eval(); ps = []; ys = []
    for x, y in loader:
        ps.append(torch.softmax(red(x.to(DISP)), 1).cpu().numpy()); ys.extend(y.tolist())
    return np.array(ys), np.concatenate(ps)

def cargadores(cfg):
    media, desv = list(cfg['mean']), list(cfg['std'])
    tf_eval = transforms.Compose([AjustarCuadrado(224), transforms.ToTensor(), transforms.Normalize(media, desv)])
    tf_train = transforms.Compose([AjustarCuadrado(224), transforms.RandomHorizontalFlip(.5),
                                   transforms.ColorJitter(brightness=.1, contrast=.1), transforms.ToTensor(),
                                   transforms.Normalize(media, desv)])
    g = torch.Generator().manual_seed(SEMILLA)
    return (DataLoader(Paneles(partes['train'], tf_train), batch_size=32, shuffle=True, num_workers=0, generator=g),
            DataLoader(Paneles(partes['val'], tf_eval), batch_size=64, num_workers=0),
            DataLoader(Paneles(partes['test'], tf_eval), batch_size=64, num_workers=0), tf_eval)

def entrenar(arquitectura, epocas=10, paciencia=3):
    torch.manual_seed(SEMILLA)
    red = timm.create_model(arquitectura, pretrained=True, num_classes=len(CLASES)).to(DISP)
    dl_tr, dl_va, dl_te, tf_eval = cargadores(red.pretrained_cfg)
    ytr = partes['train'].clase.map(CLASE_A_ID).to_numpy()
    pesos = len(ytr) / (len(CLASES) * np.bincount(ytr, minlength=len(CLASES)))
    perdida = torch.nn.CrossEntropyLoss(weight=torch.tensor(pesos, dtype=torch.float32, device=DISP))
    opt = torch.optim.AdamW(red.parameters(), lr=1e-4, weight_decay=1e-4)
    mejor, sin, estado, mejor_ep, t0 = -1, 0, None, 0, time.perf_counter()
    for ep in range(1, epocas + 1):
        red.train()
        for x, y in dl_tr:
            x, y = x.to(DISP), y.to(DISP); opt.zero_grad(set_to_none=True)
            perdida(red(x), y).backward(); opt.step()
        yv, pv = inferir(red, dl_va); f1 = metricas(yv, pv.argmax(1))['macro_f1']
        print(f"  {arquitectura} época {ep}: macro-F1 val = {f1:.3f}")
        if f1 > mejor + 1e-8:
            mejor, sin, mejor_ep = f1, 0, ep; estado = {k: v.detach().cpu().clone() for k, v in red.state_dict().items()}
        else:
            sin += 1
            if sin >= paciencia: break
    red.load_state_dict(estado)
    return red, dl_va, dl_te, tf_eval, mejor_ep, time.perf_counter() - t0

## 3. Modelo actual: EfficientNet-B0 con los pesos publicados en el repositorio
No se reentrena: se descargan los mismos pesos que usa la aplicación en Render y se evalúan
sobre la división regenerada. Si el resultado coincide con el Hito 1 (accuracy 60,7 %), queda
demostrado que la división y el preprocesamiento son los mismos.

In [ ]:
PESOS = Path("../modelo/efficientnet_b0_solar.pt")          # copia local si se ejecuta dentro del repositorio
if not PESOS.exists():                                        # en Colab se descargan desde el repositorio
    PESOS = Path("efficientnet_b0_solar.pt")
    urllib.request.urlretrieve("https://raw.githubusercontent.com/mikugonza4-gif/SolarScan/main/modelo/efficientnet_b0_solar.pt", PESOS)
eff = timm.create_model("efficientnet_b0", pretrained=False, num_classes=len(CLASES))
eff.load_state_dict(torch.load(PESOS, map_location="cpu", weights_only=True))
eff = eff.to(DISP)
cfg_eff = {'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225)}
_, dl_va_e, dl_te_e, tf_eff = cargadores(cfg_eff)

## 4. Entrenamiento de ConvNeXt-Tiny y ViT-Small con el mismo protocolo

In [ ]:
candidatos = {"EfficientNet-B0": (eff, dl_va_e, dl_te_e, tf_eff, 9, None)}
for nombre, arq in [("ConvNeXt-Tiny", "convnext_tiny.fb_in1k"), ("ViT-Small/16", "vit_small_patch16_224.augreg_in21k_ft_in1k")]:
    print("Entrenando", nombre)
    candidatos[nombre] = entrenar(arq)

## 5. Evaluación: validación (elección) y prueba (reporte)

In [ ]:
def latencia_cpu(red, tf, n=30):
    """Milisegundos por imagen en CPU con una imagen de prueba (como en Render)."""
    red_cpu = red.to("cpu").eval(); x = tf(abrir_rgb(partes['test'].ruta[0])).unsqueeze(0)
    with torch.inference_mode():
        for _ in range(3): red_cpu(x)
        t0 = time.perf_counter()
        for _ in range(n): red_cpu(x)
    ms = 1000 * (time.perf_counter() - t0) / n
    red.to(DISP); return ms

resultados, matrices = {}, {}
DANOS = [CLASE_A_ID['Electrical-damage'], CLASE_A_ID['Physical-Damage']]
for nombre, (red, dl_va, dl_te, tf, ep, seg) in candidatos.items():
    yv, pv = inferir(red, dl_va); yt, pt = inferir(red, dl_te); pred = pt.argmax(1)
    dano_real = np.isin(yt, DANOS); predice_dano = np.isin(pred, DANOS); conf = pt.max(1)
    revision = (conf < 0.70) | predice_dano
    rep = classification_report(yt, pred, labels=LABELS, output_dict=True, zero_division=0, target_names=CLASES)
    resultados[nombre] = {
        "validacion": metricas(yv, pv.argmax(1)), "prueba": metricas(yt, pred),
        "recall_por_clase": {c: round(rep[c]['recall'], 3) for c in CLASES},
        "danos_reales": int(dano_real.sum()), "dano_sin_recomendar_tecnico": int((dano_real & ~predice_dano).sum()),
        "dano_sin_derivacion_humana": int((dano_real & ~revision).sum()),
        "proporcion_revision_humana": float(revision.mean()),
        "parametros_millones": round(sum(p.numel() for p in red.parameters()) / 1e6, 2),
        "tamano_mb_fp32": round(sum(p.numel() for p in red.parameters()) * 4 / 1e6, 1),
        "latencia_cpu_ms": round(latencia_cpu(red, tf), 1), "mejor_epoca": ep,
        "segundos_entrenamiento": None if seg is None else round(seg, 1)}
    matrices[nombre] = confusion_matrix(yt, pred, labels=LABELS).tolist()

tabla = pd.DataFrame({n: {"val_macro_f1": r["validacion"]["macro_f1"], "test_accuracy": r["prueba"]["accuracy"],
                          "test_acc_bal": r["prueba"]["accuracy_balanceada"], "test_macro_f1": r["prueba"]["macro_f1"],
                          "daños sin técnico": r["dano_sin_recomendar_tecnico"],
                          "daños sin humano": r["dano_sin_derivacion_humana"],
                          "latencia CPU (ms)": r["latencia_cpu_ms"], "tamaño (MB)": r["tamano_mb_fp32"]}
                      for n, r in resultados.items()}).T
print(tabla.round(3))

# Regla de decisión fijada al inicio: se cambia el modelo en producción solo si otro supera a
# EfficientNet-B0 por al menos 0,05 de macro-F1 en validación.
base = resultados["EfficientNet-B0"]["validacion"]["macro_f1"]
mejor_otro = max((n for n in resultados if n != "EfficientNet-B0"), key=lambda n: resultados[n]["validacion"]["macro_f1"])
cambia = resultados[mejor_otro]["validacion"]["macro_f1"] >= base + 0.05
elegido = mejor_otro if cambia else "EfficientNet-B0"
print("Modelo elegido para producción:", elegido)
# Comprobación de trazabilidad: la accuracy de EfficientNet-B0 debe coincidir con la del Hito 1 (0,6068).
reproduce_hito1 = abs(resultados["EfficientNet-B0"]["prueba"]["accuracy"] - 0.6068) < 0.005
print("¿Reproduce el resultado del Hito 1?", reproduce_hito1)

In [ ]:
fig, axs = plt.subplots(1, len(matrices), figsize=(6 * len(matrices), 5))
for ax, (n, m) in zip(axs, matrices.items()):
    ax.imshow(m, cmap="Blues")
    for i in range(6):
        for j in range(6): ax.text(j, i, m[i][j], ha="center", va="center", fontsize=9)
    ax.set_xticks(range(6), CLASES, rotation=45, ha="right", fontsize=8); ax.set_yticks(range(6), CLASES, fontsize=8)
    ax.set_title(f"{n} (prueba)"); ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
plt.tight_layout(); plt.savefig(OUT / "matrices_imagenes.png", dpi=150); plt.show()

In [ ]:
salida = {"division": conteo, "resultados": resultados, "matrices_prueba": matrices,
          "regla_decision": "Cambiar solo si otro modelo supera a EfficientNet-B0 por >= 0,05 de macro-F1 en validación",
          "modelo_elegido": elegido, "reproduce_hito1": bool(reproduce_hito1), "dispositivo_entrenamiento": DISP}
(OUT / "resultados_imagenes.json").write_text(json.dumps(salida, indent=2, ensure_ascii=False))
if elegido != "EfficientNet-B0":
    red = candidatos[elegido][0].to("cpu")
    torch.save(red.state_dict(), OUT / "modelo_elegido.pt")
try:
    from google.colab import files
    files.download(str(OUT / "resultados_imagenes.json")); files.download(str(OUT / "matrices_imagenes.png"))
except ImportError:
    print("Resultados en", OUT.resolve())